In [15]:
import pandas as pd
df = pd.read_parquet(r"C:\Users\User\Documents\NUS\Y3S2\DSE3101\dse3101investmentproject\Datasets\13F_filtered_and_mapped_files\cusip_ticker_map.parquet")

In [ ]:
import openpyxl
df.to_excel(r"C:\Users\User\Documents\NUS\Y3S2\DSE3101\dse3101investmentproject\Datasets\13F_filtered_and_mapped_files\cusip_ticker_map.xlsx", index=False)

In [17]:
# ── 1. Basic shape ────────────────────────────────────────────────────────────
print("=== Basic Info ===")
print(f"Total rows:      {len(df):,}")
print(f"Unique CUSIPs:   {df['CUSIP'].nunique():,}")
print(f"Unique tickers:  {df['ticker'].nunique():,}")
print(f"Columns: {df.columns.tolist()}")
print(df.head())

print()

=== Basic Info ===
Total rows:      47,045
Unique CUSIPs:   47,045
Unique tickers:  17,350
Columns: ['CUSIP', 'ticker', 'security_type']
       CUSIP                 ticker security_type
0  98979H103                ZSANGBP  Common Stock
1  G0593M107                   None          None
2  31816QAD3  MNDT 1.625 06/01/35 B   US DOMESTIC
3  00091E109                   ABSI  Common Stock
4  964285204                   None          None



In [18]:
dupes = df[df.duplicated(subset="CUSIP", keep=False)]
print(f"Duplicate CUSIP rows: {len(dupes)}")
print(dupes.sort_values("CUSIP").head(20))

Duplicate CUSIP rows: 0
Empty DataFrame
Columns: [CUSIP, ticker, security_type]
Index: []


In [19]:
# ── 2. Mapping success rate ───────────────────────────────────────────────────
print("\n=== Mapping Rate ===")
mapped   = df["ticker"].notna().sum()
unmapped = df["ticker"].isna().sum()
total    = len(df)
print(f"Mapped:   {mapped:,} ({mapped/total:.1%})")
print(f"Unmapped: {unmapped:,} ({unmapped/total:.1%})")


=== Mapping Rate ===
Mapped:   17,471 (37.1%)
Unmapped: 29,574 (62.9%)


In [20]:
# ── 3. Duplicates — CUSIPs mapping to multiple tickers ───────────────────────
print("\n=== Duplicate CUSIPs (one CUSIP -> multiple tickers) ===")
print(df.groupby("CUSIP")["ticker"].count().value_counts().rename("num_CUSIPs"))


=== Duplicate CUSIPs (one CUSIP -> multiple tickers) ===
ticker
0    29574
1    17471
Name: num_CUSIPs, dtype: int64


In [21]:
# ── 4. Security type breakdown (if column exists) ─────────────────────────────
if "security_type" in df.columns:
    print("\n=== Security Type Breakdown ===")
    print(df["security_type"].value_counts(dropna=False).head(15))

    print("\n=== Mapped vs Unmapped by Security Type ===")
    print(df.groupby("security_type")["ticker"].apply(
        lambda x: f"mapped={x.notna().sum()}, unmapped={x.isna().sum()}"
    ))


=== Security Type Breakdown ===
security_type
None               29575
Common Stock        7944
ETP                 2851
Open-End Fund       1723
ADR                 1309
PUBLIC               826
US DOMESTIC          538
Closed-End Fund      471
GLOBAL               299
REIT                 294
FIXED                253
FIXED, OID           185
DOMESTIC MTN         149
Equity WRT           107
US GOVERNMENT        102
Name: count, dtype: int64

=== Mapped vs Unmapped by Security Type ===
security_type
ADJUSTABLE             mapped=1, unmapped=0
ADR                 mapped=1309, unmapped=0
Agncy ABS Other        mapped=1, unmapped=0
Agncy CMO Other        mapped=5, unmapped=0
CANADIAN               mapped=1, unmapped=0
Closed-End Fund      mapped=471, unmapped=0
Common Stock        mapped=7944, unmapped=0
DOMESTIC               mapped=9, unmapped=0
DOMESTIC MTN         mapped=149, unmapped=0
ETP                 mapped=2851, unmapped=0
EURO NON-DOLLAR        mapped=1, unmapped=0
EURO-DOLL

In [22]:
# ── 5. Ticker frequency — most held securities ───────────────────────────────
print("\n=== Top 20 Most Frequent Tickers ===")
print(df["ticker"].value_counts().head(20))
# if AAPL, MSFT, GOOGL dominate → good sign, these are held via many CUSIPs


=== Top 20 Most Frequent Tickers ===
ticker
BZQUSD                    3
FAZEUR                    3
SKF                       3
DUSTUSD                   3
SOXS                      3
CIA                       2
SLG2EUR                   2
WPG2EUR                   2
NVIVDEUR                  2
CHKEUR                    2
ERY                       2
KSCP                      2
YJ                        2
LYRA                      2
TZAEUR                    2
MO MOSENV 5 07/01/2024    2
ABVC                      2
SERVUSD                   2
SFUNUSD                   2
CPTAUSD                   2
Name: count, dtype: int64


In [23]:
# ── 6. Suspicious tickers ─────────────────────────────────────────────────────
print("\n=== Suspiciously Short/Long Tickers ===")
ticker_lengths = df["ticker"].dropna().str.len()
print(ticker_lengths.value_counts().sort_index())
# Most legitimate tickers are 1-5 chars; flag anything longer

suspicious = df[df["ticker"].dropna().str.len() > 6]
print(f"Tickers longer than 6 chars: {len(suspicious)}")
print(suspicious["ticker"].value_counts().head(10))



=== Suspiciously Short/Long Tickers ===
ticker
1       19
2      241
3     2500
4     6509
5     3812
6      556
7     1260
8      146
9        6
10       5
11      15
12     101
13      86
14     167
15     259
16     353
17     309
18     309
19     289
20      82
21      65
22     270
23      10
24      26
25      73
26       3
Name: count, dtype: int64


C:\Users\User\AppData\Local\Temp\ipykernel_7232\1672006808.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  suspicious = df[df["ticker"].dropna().str.len() > 6]


IndexingError: Unalignable boolean Series provided as indexer (index of the boolean Series and of the indexed object do not match).

In [24]:
# ── 7. Final clean count ──────────────────────────────────────────────────────
if "security_type" in df.columns:
    common_stock = df[
        (df["security_type"] == "Common Stock") &
        (df["ticker"].notna())
    ]
    print(f"\n=== Common Stock Only ===")
    print(f"Unique CUSIPs:  {common_stock['CUSIP'].nunique():,}")
    print(f"Unique tickers: {common_stock['ticker'].nunique():,}")


=== Common Stock Only ===
Unique CUSIPs:  7,944
Unique tickers: 7,885
